# Tutorial: BM25 Retrieval and Ranking (End-to-End)

**Audience**
- Engineers learning lexical retrieval and search ranking.

**Prerequisites**
- Basic Python and dictionaries/lists.
- Familiarity with tokenization and term frequency.

**Learning goals**
- Implement BM25 from scratch.
- Rank documents for user queries.
- Inspect per-term score contributions.
- Understand how `k1` and `b` affect ranking.


## Outline

1. Setup a sample corpus and queries.
2. Tokenize text and compute corpus statistics.
3. Build a BM25 scorer from first principles.
4. Run retrieval and ranking.
5. Explain score contributions for interpretability.
6. Test multiple queries and tune hyperparameters.


In [1]:
from __future__ import annotations

import math
import re
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Tuple

# Small in-memory corpus with diverse topics and lengths.
corpus = {
    "D1": "Python programming for data science and machine learning.",
    "D2": "A practical guide to cooking Italian pasta with tomato and basil.",
    "D3": "Advanced ranking models for search and information retrieval.",
    "D4": "How to train for a marathon with weekly running plans.",
    "D5": "Building retrieval systems using BM25 and vector embeddings.",
    "D6": "Data engineering pipelines for reliable analytics and reporting.",
    "D7": "Neural search combines lexical retrieval with dense retrieval.",
    "D8": "Beginner tutorial for Python and data analysis with pandas.",
}

queries = [
    "python data retrieval",
    "search ranking bm25",
    "italian pasta guide",
]

print(f"Documents: {len(corpus)}")
print(f"Queries:   {len(queries)}")

Documents: 8
Queries:   3


## Step 1: Preprocess Text

We lowercase and tokenize by alphanumeric terms. This keeps implementation simple and deterministic.


In [2]:
def tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

# Tokenize the full corpus.
tokenized_corpus: Dict[str, List[str]] = {doc_id: tokenize(text) for doc_id, text in corpus.items()}

doc_lengths = {doc_id: len(tokens) for doc_id, tokens in tokenized_corpus.items()}

print("Tokenized documents (first 3 shown):")
for doc_id in list(tokenized_corpus)[:3]:
    print(f"{doc_id}: {tokenized_corpus[doc_id]}")

print("\nDocument lengths:")
for doc_id, length in doc_lengths.items():
    print(f"{doc_id}: {length}")

Tokenized documents (first 3 shown):
D1: ['python', 'programming', 'for', 'data', 'science', 'and', 'machine', 'learning']
D2: ['a', 'practical', 'guide', 'to', 'cooking', 'italian', 'pasta', 'with', 'tomato', 'and', 'basil']
D3: ['advanced', 'ranking', 'models', 'for', 'search', 'and', 'information', 'retrieval']

Document lengths:
D1: 8
D2: 11
D3: 8
D4: 10
D5: 8
D6: 8
D7: 8
D8: 9


## Step 2: Compute BM25 Corpus Statistics

BM25 needs:
- `N`: number of documents
- `avgdl`: average document length
- `df(term)`: document frequency per term
- `idf(term)`: inverse document frequency

We use Robertson/Sparck Jones style IDF with +0.5 smoothing:

\[
	ext{idf}(t) = \log\left(1 + 
rac{N - df(t) + 0.5}{df(t) + 0.5}
ight)
\]


In [13]:
N = len(tokenized_corpus)
avgdl = sum(len(tokens) for tokens in tokenized_corpus.values()) / N

# df: in how many documents each term appears.
df = Counter()
for tokens in tokenized_corpus.values():
    for term in set(tokens):
        df[term] += 1

idf = {
    term: math.log(1.0 + (N - freq + 0.5) / (freq + 0.5))
    for term, freq in df.items()
}

print(f"N = {N}")
print(f"avgdl = {avgdl:.3f}")

print("\nSample IDF values:")
for term in ["python", "data", "retrieval", "bm25", "pasta", "search"]:
    print(f"idf({term:9s}) = {idf.get(term, 0.0):.4f}")

N = 8
avgdl = 8.750

Sample IDF values:
idf(python   ) = 1.2809
idf(data     ) = 0.9445
idf(retrieval) = 0.9445
idf(bm25     ) = 1.7918
idf(pasta    ) = 1.7918
idf(search   ) = 1.2809


## Step 3: Implement BM25 Scoring

For query term `t` and document `d`:

\[
	ext{score}(d, q) = \sum_{t \in q} 	ext{idf}(t) \cdot 
rac{tf(t,d)(k_1+1)}{tf(t,d) + k_1\left(1-b+b
rac{|d|}{avgdl}
ight)}
\]

- `k1` controls term-frequency saturation.
- `b` controls document-length normalization.


In [4]:
@dataclass
class BM25:
    tokenized_corpus: Dict[str, List[str]]
    idf: Dict[str, float]
    avgdl: float
    k1: float = 1.5
    b: float = 0.75

    def score_doc(self, query_tokens: List[str], doc_id: str) -> float:
        terms = Counter(self.tokenized_corpus[doc_id])
        dl = len(self.tokenized_corpus[doc_id])
        score = 0.0

        for term in query_tokens:
            if term not in terms:
                continue
            tf = terms[term]
            term_idf = self.idf.get(term, 0.0)
            denom = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            score += term_idf * (tf * (self.k1 + 1)) / denom

        return score

    def rank(self, query: str, top_k: int = 5) -> List[Tuple[str, float]]:
        query_tokens = tokenize(query)
        scored = [
            (doc_id, self.score_doc(query_tokens, doc_id))
            for doc_id in self.tokenized_corpus
        ]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

    def explain(self, query: str, doc_id: str) -> List[Tuple[str, float]]:
        query_tokens = tokenize(query)
        terms = Counter(self.tokenized_corpus[doc_id])
        dl = len(self.tokenized_corpus[doc_id])
        contribs = []

        for term in query_tokens:
            tf = terms.get(term, 0)
            if tf == 0:
                contribs.append((term, 0.0))
                continue
            term_idf = self.idf.get(term, 0.0)
            denom = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            part = term_idf * (tf * (self.k1 + 1)) / denom
            contribs.append((term, part))
        return contribs

bm25 = BM25(tokenized_corpus=tokenized_corpus, idf=idf, avgdl=avgdl)
print(f"BM25 configured with k1={bm25.k1}, b={bm25.b}")

BM25 configured with k1=1.5, b=0.75


## Step 4: Rank Documents for a Query

Now we run end-to-end retrieval and ranking.


In [5]:
query = "python data retrieval"
ranking = bm25.rank(query, top_k=8)

print(f"Query: {query!r}\n")
for i, (doc_id, score) in enumerate(ranking, start=1):
    print(f"{i:>2}. {doc_id}  score={score:.4f}  text={corpus[doc_id]}")

Query: 'python data retrieval'

 1. D1  score=2.3147  text=Python programming for data science and machine learning.
 2. D8  score=2.1971  text=Beginner tutorial for Python and data analysis with pandas.
 3. D7  score=1.3875  text=Neural search combines lexical retrieval with dense retrieval.
 4. D3  score=0.9824  text=Advanced ranking models for search and information retrieval.
 5. D5  score=0.9824  text=Building retrieval systems using BM25 and vector embeddings.
 6. D6  score=0.9824  text=Data engineering pipelines for reliable analytics and reporting.
 7. D2  score=0.0000  text=A practical guide to cooking Italian pasta with tomato and basil.
 8. D4  score=0.0000  text=How to train for a marathon with weekly running plans.


## Step 5: Explain Why the Top Result Won

We inspect per-term contributions for the top-ranked document.


In [6]:
top_doc = ranking[0][0]
contribs = bm25.explain(query, top_doc)

print(f"Top doc: {top_doc}")
print(corpus[top_doc])
print("\nTerm contributions:")
for term, value in contribs:
    print(f"- {term:10s}: {value:.4f}")
print(f"Total score: {sum(v for _, v in contribs):.4f}")

Top doc: D1
Python programming for data science and machine learning.

Term contributions:
- python    : 1.3323
- data      : 0.9824
- retrieval : 0.0000
Total score: 2.3147


## Step 6: Batch Retrieval for Multiple Queries

This mirrors real search behavior where many user queries are evaluated.


In [7]:
for q in queries:
    top3 = bm25.rank(q, top_k=3)
    print(f"\nQuery: {q!r}")
    for rank_i, (doc_id, score) in enumerate(top3, start=1):
        print(f"  {rank_i}. {doc_id} ({score:.4f})")


Query: 'python data retrieval'
  1. D1 (2.3147)
  2. D8 (2.1971)
  3. D7 (1.3875)

Query: 'search ranking bm25'
  1. D3 (3.1960)
  2. D5 (1.8636)
  3. D7 (1.3323)

Query: 'italian pasta guide'
  1. D2 (4.8178)
  2. D1 (0.0000)
  3. D3 (0.0000)


## Step 7: Hyperparameter Sensitivity (`k1`, `b`)

We vary parameters to show how ranking shifts.


In [8]:
q = "search retrieval ranking"
configs = [
    (0.8, 0.3),
    (1.2, 0.75),
    (2.0, 0.9),
]

for k1, b in configs:
    model = BM25(tokenized_corpus=tokenized_corpus, idf=idf, avgdl=avgdl, k1=k1, b=b)
    best_doc, best_score = model.rank(q, top_k=1)[0]
    print(f"k1={k1:<3} b={b:<4} -> top1={best_doc}, score={best_score:.4f}")

k1=0.8 b=0.3  -> top1=D3, score=4.0636
k1=1.2 b=0.75 -> top1=D3, score=4.1631
k1=2.0 b=0.9  -> top1=D3, score=4.2350


## Common Pitfall and Extension

**Pitfall**
- Using inconsistent tokenization between indexing and querying causes silent recall loss.

**Extension**
- Add stopword filtering and stemming, then compare top-k results before/after.


## Exercise

Add one new document heavily focused on `retrieval`, then rerun ranking for query `"python data retrieval"`.
Predict which document should move up and verify with code.


In [9]:
# Exercise answer scaffold
# 1) Add a new document to `corpus`
# 2) Recompute tokenized_corpus, df, idf, avgdl
# 3) Rebuild BM25 and print new ranking
pass

In [10]:
# One possible solution
extended_corpus = dict(corpus)
extended_corpus["D9"] = "Python retrieval retrieval retrieval for search ranking and data systems."

extended_tokens = {doc_id: tokenize(text) for doc_id, text in extended_corpus.items()}
N2 = len(extended_tokens)
avgdl2 = sum(len(toks) for toks in extended_tokens.values()) / N2

df2 = Counter()
for toks in extended_tokens.values():
    for term in set(toks):
        df2[term] += 1

idf2 = {
    term: math.log(1.0 + (N2 - freq + 0.5) / (freq + 0.5))
    for term, freq in df2.items()
}

bm25_v2 = BM25(tokenized_corpus=extended_tokens, idf=idf2, avgdl=avgdl2)
new_ranking = bm25_v2.rank("python data retrieval", top_k=5)

print("Top 5 after adding D9:")
for i, (doc_id, score) in enumerate(new_ranking, start=1):
    print(f"{i}. {doc_id} ({score:.4f})")

Top 5 after adding D9:
1. D9 (3.0404)
2. D1 (1.9354)
3. D8 (1.8380)
4. D7 (1.1786)
5. D3 (0.8361)
